# Experiment 1: Image Generation

**Experiment:** Exp 1

Conditional image generation with StyleGAN2-ADA (StyleGAN-C). Generates class-conditional synthetic MRI slices and visualizes results (Figures 5.1-5.5).

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
!pip install gdown --upgrade

if os.path.isdir("/content/drive/MyDrive/colab-sg2-ada-pytorch"):
    %cd "/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch"
elif os.path.isdir("/content/drive/"):
    #install script
    %cd "/content/drive/MyDrive/"
    !mkdir colab-sg2-ada-pytorch
    %cd colab-sg2-ada-pytorch
    !git clone https://github.com/JunTierSS/stylegan2-ada
    %cd stylegan2-ada-pytorch
    !mkdir downloads
    !mkdir datasets
    !mkdir pretrained
    !gdown --id 1-5xZkD8ajXw1DdopTkH_rAoCsD72LhKU -O /content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/pretrained/wikiart.pkl
else:
    !git clone https://github.com/JunTierSS/stylegan2-ada
    %cd stylegan2-ada-pytorch
    !mkdir downloads
    !mkdir datasets
    !mkdir pretrained
    %cd pretrained
    !gdown --id 1-5xZkD8ajXw1DdopTkH_rAoCsD72LhKU
    %cd ../

In [ ]:
pip install ninja

In [ ]:
pip install ninja opensimplex

In [ ]:
# generate_grid_3x5.py
# Genera una grilla 3x5 de imágenes condicionales (glioma, meningioma, pituitary)
# usando un StyleGAN2-ADA condicional entrenado en MRIs.

import numpy as np
import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# IMPORTS DEL REPO STYLEGAN2-ADA (oficial NVidia)
from legacy import load_network_pkl


def to_uint8(img_tensor: torch.Tensor) -> np.ndarray:
    """
    Convierte imagen [-1,1] (1,C,H,W) o (C,H,W) a uint8 para matplotlib/PIL.
    """
    if img_tensor.dim() == 4:
        img = img_tensor[0]
    else:
        img = img_tensor
    img = (img.clamp(-1, 1) + 1) * 0.5  # [0,1]
    img = (img * 255.0).clamp(0, 255).to(torch.uint8)

    # [C,H,W] -> [H,W] o [H,W,3]
    if img.shape[0] == 1:
        img = img[0]  # [H,W]
    else:
        img = img.permute(1, 2, 0)  # [H,W,C]
    return img.cpu().numpy()


def generate_grid_3x5_conditional(
    network_pkl: str,
    out_path: str = "grid_3x5_conditional.png",
    class_names = ("Glioma", "Meningioma", "Pituitary"),
    num_rows: int = 3,
    truncation_psi: float = 1.4,
    seed: int = 42,
    show: bool = True,
):
    """
    Genera una grilla de tamaño (num_rows x len(class_names)) y la guarda en out_path.
    Si show=True, también muestra la imagen en pantalla.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] Using {device}")

    # ---------- Cargar G_ema ----------
    print(f"[Load] Loading generator from: {network_pkl}")
    with open(network_pkl, "rb") as f:
        data = load_network_pkl(f)
    G = data.get("G_ema", None) or data.get("G", None)
    if G is None:
        raise RuntimeError("No se encontró G_ema ni G en el .pkl")
    G = G.to(device).eval()

    c_dim = getattr(G, "c_dim", 0)
    z_dim = getattr(G, "z_dim", 512)
    if c_dim <= 0:
        raise RuntimeError("El generador no es condicional (c_dim=0).")

    # ---------- Preparar figura ----------
    n_classes = len(class_names)
    fig, axes = plt.subplots(
        nrows=num_rows, ncols=n_classes,
        figsize=(3 * n_classes, 3 * num_rows)
    )

    # Normalizar shape de axes
    if num_rows == 1 and n_classes == 1:
        axes = np.array([[axes]])
    elif num_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    elif n_classes == 1:
        axes = np.expand_dims(axes, axis=1)

    rng = np.random.RandomState(seed)

    # ---------- Generación ----------
    with torch.no_grad():
        for row in range(num_rows):
            # mismo z para todas las clases de la fila
            z = torch.from_numpy(rng.randn(1, z_dim)).to(device)

            for col, cname in enumerate(class_names):
                ci = col  # asumimos orden 0:glioma, 1:meningioma, 2:pituitary
                c = torch.zeros(1, c_dim, device=device)
                c[0, ci] = 1.0

                try:
                    img = G(z, c, truncation_psi=float(truncation_psi), noise_mode="const")
                except TypeError:
                    img = G(z, c, truncation_psi=float(truncation_psi))

                img_np = to_uint8(img)

                ax = axes[row, col]
                if img_np.ndim == 2:
                    ax.imshow(img_np, cmap="gray")
                else:
                    ax.imshow(img_np)
                ax.axis("off")

                # Títulos solo en la primera fila
                if row == 0:
                    ax.set_title(cname, fontsize=16)

    plt.tight_layout()

    # ---------- Guardar ----------
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"[OK] Saved grid to: {out_path}")

    # ---------- Mostrar ----------
    if show:
        plt.show()

    plt.close(fig)

#completo

In [ ]:

def main():
    # Config por defecto: cámbialos a tu gusto o pásalos desde otro script
    network_pkl ="/content/drive/MyDrive/TESIS/StyleGan2/Models/00003-finalnuevo-cond-mirror-24gb-gpu-kimg1000-batch32-ada-target0.6-bg/network-snapshot-000900.pkl" # <-- modifica aquí
    out_path = "grid_3x5_conditional.png"
    generate_grid_3x5_conditional(
        network_pkl=network_pkl,
        out_path=out_path,
        class_names=("Glioma 1.0", "Meningioma 1.0", "Pituitary 1.0"),
        num_rows=3,
        truncation_psi=1.0,
        seed=886,
        show=True,   # muestra la imagen además de guardarla
    )


if __name__ == "__main__":
    main()

#inge

In [ ]:

def main():
    # Config por defecto: cámbialos a tu gusto o pásalos desde otro script
    network_pkl = "/content/drive/MyDrive/TESIS/StyleGan2/Modelsinge/00000-finalnuevoinge-cond-mirror-24gb-gpu-kimg1000-batch32-ada-target0.6-bg/network-snapshot-000600.pkl"
    out_path = "grid_3x5_conditional.png"
    generate_grid_3x5_conditional(
        network_pkl=network_pkl,
        out_path=out_path,
        class_names=("Glioma 1.0", "Meningioma 1.0", "Pituitary 1.0"),
        num_rows=3,
        truncation_psi=1.0,
        seed=9381,
        show=True,   # muestra la imagen además de guardarla
    )


if __name__ == "__main__":
    main()

In [ ]:
# compare_real_fake_2x6.py
# 2 filas x 6 columnas:
#   fila 0: imágenes sintéticas (GAN)
#   fila 1: imágenes reales
#   2 columnas por clase (glioma, meningioma, pituitary)

import os
from pathlib import Path

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from legacy import load_network_pkl   # del repo StyleGAN2-ADA

# ---------------- CONFIG ----------------
NETWORK_PKL = "/content/drive/MyDrive/TESIS/StyleGan2/Models/00003-finalnuevo-cond-mirror-24gb-gpu-kimg1000-batch32-ada-target0.6-bg/network-snapshot-000900.pkl"   # <-- CAMBIA ESTO
DATA_ROOT   = "/content/localdata/train_unzipped"               # <-- root/glioma/*, root/meningioma/*, root/pituitary/*
OUT_PATH    = "grid_real_vs_fake_2x6.png"
TRUNCATION_PSI = 1.0
SEED = 42

CLASS_NAMES = ["glioma", "meningioma", "pituitary"]  # orden de las columnas
N_PER_CLASS = 2                                      # 2 imágenes por clase
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
# ----------------------------------------


def to_uint8(img_tensor: torch.Tensor) -> np.ndarray:
    """Convierte imagen [-1,1] (1,C,H,W) o (C,H,W) a uint8 para matplotlib/PIL."""
    if img_tensor.dim() == 4:
        img = img_tensor[0]
    else:
        img = img_tensor
    img = (img.clamp(-1, 1) + 1) * 0.5  # [0,1]
    img = (img * 255.0).clamp(0, 255).to(torch.uint8)
    if img.shape[0] == 1:
        img = img[0]              # [H,W]
    else:
        img = img.permute(1, 2, 0)  # [H,W,C]
    return img.cpu().numpy()


def pick_real_images(data_root: str | Path,
                     class_names,
                     n_per_class: int,
                     rng: np.random.RandomState):
    """Selecciona n_per_class imágenes reales por clase."""
    data_root = Path(data_root)
    real_paths = {}
    for cname in class_names:
        cdir = data_root / cname
        if not cdir.is_dir():
            raise FileNotFoundError(f"No se encontró carpeta real para clase '{cname}': {cdir}")
        imgs = [p for p in cdir.rglob("*")
                if p.is_file() and p.suffix.lower() in IMG_EXTS]
        if len(imgs) < n_per_class:
            raise RuntimeError(f"Clase '{cname}' solo tiene {len(imgs)} imágenes reales (< {n_per_class}).")
        rng.shuffle(imgs)
        real_paths[cname] = imgs[:n_per_class]
    return real_paths


def generate_real_fake_grid_2x6(
    network_pkl: str = NETWORK_PKL,
    data_root: str | Path = DATA_ROOT,
    out_path: str | Path = OUT_PATH,
    truncation_psi: float = TRUNCATION_PSI,
    seed: int = SEED,
    class_names = CLASS_NAMES,
    n_per_class: int = N_PER_CLASS,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] Using {device}")

    # ---------- cargar G_ema ----------
    print(f"[Load] Loading generator from: {network_pkl}")
    with open(network_pkl, "rb") as f:
        data = load_network_pkl(f)
    G = data.get("G_ema", None) or data.get("G", None)
    if G is None:
        raise RuntimeError("No se encontró G_ema ni G en el .pkl")
    G = G.to(device).eval()

    c_dim = getattr(G, "c_dim", 0)
    z_dim = getattr(G, "z_dim", 512)
    if c_dim <= 0:
        raise RuntimeError("El generador no es condicional (c_dim=0).")

    rng = np.random.RandomState(seed)

    # ---------- elegir imágenes reales ----------
    real_paths = pick_real_images(data_root, class_names, n_per_class, rng)

    n_classes = len(class_names)
    n_cols = n_classes * n_per_class  # 3 * 2 = 6

    fig, axes = plt.subplots(
        nrows=2, ncols=n_cols,
        figsize=(2.5 * n_cols, 5),  # ajusta si quieres
    )

    # Por si matplotlib devuelve ejes 1D
    if n_cols == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    # ---------- generar fila sintética + fila real ----------
    with torch.no_grad():
        for ci, cname in enumerate(class_names):
            for k in range(n_per_class):
                col = ci * n_per_class + k

                # --- SINTÉTICA (fila 0) ---
                z = torch.from_numpy(rng.randn(1, z_dim)).to(device)
                c = torch.zeros(1, c_dim, device=device)
                c[0, ci] = 1.0
                try:
                    img_fake = G(z, c, truncation_psi=truncation_psi, noise_mode="const")
                except TypeError:
                    img_fake = G(z, c, truncation_psi=truncation_psi)
                fake_np = to_uint8(img_fake)

                ax_fake = axes[0, col]
                if fake_np.ndim == 2:
                    ax_fake.imshow(fake_np, cmap="gray")
                else:
                    ax_fake.imshow(fake_np)
                ax_fake.axis("off")

                # --- REAL (fila 1) ---
                real_path = real_paths[cname][k]
                img_real = Image.open(real_path).convert("L")
                real_np = np.array(img_real)

                ax_real = axes[1, col]
                ax_real.imshow(real_np, cmap="gray")
                ax_real.axis("off")

                # títulos solo en fila superior
                ax_fake.set_title(f"{cname.capitalize()} #{k+1}", fontsize=12)

    # etiquetas de filas
    axes[0, 0].set_ylabel("Synthetic", fontsize=14)
    axes[1, 0].set_ylabel("Real", fontsize=14)

    plt.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()  # "imprime" la imagen en pantalla
    plt.close(fig)
    print(f"[OK] Saved grid to: {out_path}")


if __name__ == "__main__":
    generate_real_fake_grid_2x6()

In [ ]:
# compare_real_fake_2x6.py
# 2 filas x 6 columnas:
#   fila 0: imágenes sintéticas (GAN)
#   fila 1: imágenes reales
#   2 columnas por clase (glioma, meningioma, pituitary)

import os
from pathlib import Path

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from legacy import load_network_pkl   # del repo StyleGAN2-ADA

# ---------------- CONFIG ----------------
NETWORK_PKL = "/content/drive/MyDrive/TESIS/StyleGan2/Models/00003-finalnuevo-cond-mirror-24gb-gpu-kimg1000-batch32-ada-target0.6-bg/network-snapshot-000900.pkl"
DATA_ROOT   = "/content/localdata/train_unzipped"   # root/glioma/*, root/meningioma/*, root/pituitary/*
OUT_PATH    = "grid_real_vs_fake_2x6.png"
TRUNCATION_PSI = 1.0
SEED = 42

CLASS_NAMES = ["glioma", "meningioma", "pituitary"]  # orden de las columnas
N_PER_CLASS = 2                                      # 2 imágenes por clase
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
# ----------------------------------------


def to_uint8(img_tensor: torch.Tensor) -> np.ndarray:
    """Convierte imagen [-1,1] (1,C,H,W) o (C,H,W) a uint8 para matplotlib/PIL."""
    if img_tensor.dim() == 4:
        img = img_tensor[0]
    else:
        img = img_tensor
    img = (img.clamp(-1, 1) + 1) * 0.5  # [0,1]
    img = (img * 255.0).clamp(0, 255).to(torch.uint8)
    if img.shape[0] == 1:
        img = img[0]              # [H,W]
    else:
        img = img.permute(1, 2, 0)  # [H,W,C]
    return img.cpu().numpy()


def pick_real_images(data_root: str | Path,
                     class_names,
                     n_per_class: int,
                     rng: np.random.RandomState):
    """Selecciona n_per_class imágenes reales por clase."""
    data_root = Path(data_root)
    real_paths = {}
    for cname in class_names:
        cdir = data_root / cname
        if not cdir.is_dir():
            raise FileNotFoundError(f"No se encontró carpeta real para clase '{cname}': {cdir}")
        imgs = [p for p in cdir.rglob("*")
                if p.is_file() and p.suffix.lower() in IMG_EXTS]
        if len(imgs) < n_per_class:
            raise RuntimeError(f"Clase '{cname}' solo tiene {len(imgs)} imágenes reales (< {n_per_class}).")
        rng.shuffle(imgs)
        real_paths[cname] = imgs[:n_per_class]
    return real_paths


def generate_real_fake_grid_2x6(
    network_pkl: str = NETWORK_PKL,
    data_root: str | Path = DATA_ROOT,
    out_path: str | Path = OUT_PATH,
    truncation_psi: float = TRUNCATION_PSI,
    seed: int = SEED,
    class_names = CLASS_NAMES,
    n_per_class: int = N_PER_CLASS,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] Using {device}")

    # ---------- cargar G_ema ----------
    print(f"[Load] Loading generator from: {network_pkl}")
    with open(network_pkl, "rb") as f:
        data = load_network_pkl(f)
    G = data.get("G_ema", None) or data.get("G", None)
    if G is None:
        raise RuntimeError("No se encontró G_ema ni G en el .pkl")
    G = G.to(device).eval()

    c_dim = getattr(G, "c_dim", 0)
    z_dim = getattr(G, "z_dim", 512)
    if c_dim <= 0:
        raise RuntimeError("El generador no es condicional (c_dim=0).")

    rng = np.random.RandomState(seed)

    # ---------- elegir imágenes reales ----------
    real_paths = pick_real_images(data_root, class_names, n_per_class, rng)

    n_classes = len(class_names)
    n_cols = n_classes * n_per_class  # 3 * 2 = 6

    fig, axes = plt.subplots(
        nrows=2, ncols=n_cols,
        figsize=(2.5 * n_cols, 5),
    )

    # Por si matplotlib devuelve ejes 1D
    if n_cols == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    # ---------- generar fila sintética + fila real ----------
    with torch.no_grad():
        for ci, cname in enumerate(class_names):
            for k in range(n_per_class):
                col = ci * n_per_class + k

                # --- SINTÉTICA (fila 0) ---
                z = torch.from_numpy(rng.randn(1, z_dim)).to(device)
                c = torch.zeros(1, c_dim, device=device)
                c[0, ci] = 1.0
                try:
                    img_fake = G(z, c, truncation_psi=truncation_psi, noise_mode="const")
                except TypeError:
                    img_fake = G(z, c, truncation_psi=truncation_psi)
                fake_np = to_uint8(img_fake)

                ax_fake = axes[0, col]
                if fake_np.ndim == 2:
                    ax_fake.imshow(fake_np, cmap="gray")
                else:
                    ax_fake.imshow(fake_np)
                ax_fake.axis("off")

                # --- REAL (fila 1) ---
                real_path = real_paths[cname][k]
                img_real = Image.open(real_path).convert("L")
                real_np = np.array(img_real)

                ax_real = axes[1, col]
                ax_real.imshow(real_np, cmap="gray")
                ax_real.axis("off")

                # títulos solo en fila superior
                ax_fake.set_title(f"{cname.capitalize()} #{k+1}", fontsize=12)

    # ---------- labels en eje Y (por fila, centrados a la izquierda) ----------
    fig.text(0.02, 0.75, "Synthetic", va="center", ha="center",
             rotation="vertical", fontsize=14)
    fig.text(0.02, 0.25, "Real", va="center", ha="center",
             rotation="vertical", fontsize=14)

    plt.tight_layout(rect=(0.05, 0.0, 1.0, 1.0))  # deja margen para los labels Y
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"[OK] Saved grid to: {out_path}")


if __name__ == "__main__":
    generate_real_fake_grid_2x6()

In [ ]:
import sys

print("Versiones de las librerías utilizadas:")
print(f"Python: {sys.version.split(' ')[0]}")

try:
    import gdown
    print(f"gdown: {gdown.__version__}")
except ImportError:
    print("gdown: No instalada")

try:
    import numpy as np
    print(f"numpy: {np.__version__}")
except ImportError:
    print("numpy: No instalada")

try:
    import torch
    print(f"torch: {torch.__version__}")
except ImportError:
    print("torch: No instalada")

try:
    import PIL
    print(f"Pillow (PIL): {PIL.__version__}")
except ImportError:
    print("Pillow (PIL): No instalada")

try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("matplotlib: No instalada")

try:
    import ninja
    print(f"ninja: {ninja.__version__}")
except ImportError:
    print("ninja: No instalada")

try:
    import opensimplex
    print(f"opensimplex: {opensimplex.__version__}")
except ImportError:
    print("opensimplex: No instalada")

print("\nLas librerías 'os', 'shutil', 'pathlib' y 'subprocess' son módulos estándar de Python y no tienen un atributo __version__.")
print("El módulo 'legacy' es parte del repositorio StyleGAN2-ADA y no una librería externa con versión independiente.")

# Task
I will now proceed with defining the `generate_class_truncation_grid` function and then calling it with the specified parameters to generate the grid. This involves:

1.  **Defining the `generate_class_truncation_grid` function**: This function will take the network path, output path, class names, truncation values, seed, and a show flag as input.
2.  **Implementing the grid generation logic within the function**: This will include loading the StyleGAN2-ADA generator, mapping class names to indices, iterating through classes and truncation values to generate images, and arranging them in a matplotlib grid. It will also add appropriate titles and labels.
3.  **Calling the function**: The function will be called with the provided `network_pkl`, `out_path`, `class_names=["Glioma", "Meningioma", "Pituitary"]`, `truncation_psis=[0.7, 0.9, 1.1, 1.3]`, `seed`, and `show=True`.

## Define generate_class_truncation_grid function

### Subtask:
Define a new Python function `generate_class_truncation_grid` that takes `network_pkl`, `out_path`, `class_names`, `truncation_psis`, `seed`, and `show` as arguments. This function will be responsible for creating the grid with classes as rows and truncation values as columns.


**Reasoning**:
The user wants to define a new Python function with specific parameters and a docstring. I will create a `code_block` with the function signature and a comprehensive docstring, leaving the function body to be filled in subsequent steps.



In [ ]:
def generate_class_truncation_grid(
    network_pkl: str,
    out_path: str = "grid_class_truncation.png",
    class_names: list[str] = ("Glioma", "Meningioma", "Pituitary"),
    truncation_psis: list[float] = [0.7, 0.8,0.9, 1.0, 1.1 ,1.2, 1.3],
    seed: int = 42,
    show: bool = True,
):
    """
    Generates a grid of images where rows represent different classes
    and columns represent different truncation_psi values.
    The grid is saved to `out_path` and optionally displayed.

    Args:
        network_pkl (str): Path to the StyleGAN2-ADA network pickle file.
        out_path (str): Path to save the generated grid image.
        class_names (list[str]): List of class names to generate images for.
                                 Each class will correspond to a row in the grid.
        truncation_psis (list[float]): List of truncation psi values.
                                       Each value will correspond to a column in the grid.
        seed (int): Random seed for reproducibility.
        show (bool): If True, displays the generated image grid using matplotlib.
    """
    pass

## Implement grid generation logic

### Subtask:
Inside the `generate_class_truncation_grid` function, implement the logic to load the StyleGAN2-ADA generator, map class names to indices, create a matplotlib grid, generate images for each class and truncation value, plot them, add titles and labels, and then save and display the grid.


**Reasoning**:
I will implement the logic for the `generate_class_truncation_grid` function as per the instructions, including loading the StyleGAN2-ADA generator, preparing the matplotlib grid, generating and plotting images, and adding appropriate titles and labels. I will also include necessary imports and the `to_uint8` helper function within this code block to make it self-contained.



In [ ]:
import numpy as np
import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# IMPORTS DEL REPO STYLEGAN2-ADA (oficial NVidia)
from legacy import load_network_pkl

def to_uint8(img_tensor: torch.Tensor) -> np.ndarray:
    """
    Convierte imagen [-1,1] (1,C,H,W) o (C,H,W) a uint8 para matplotlib/PIL.
    """
    if img_tensor.dim() == 4:
        img = img_tensor[0]
    else:
        img = img_tensor
    img = (img.clamp(-1, 1) + 1) * 0.5  # [0,1]
    img = (img * 255.0).clamp(0, 255).to(torch.uint8)

    # [C,H,W] -> [H,W] o [H,W,3]
    if img.shape[0] == 1:
        img = img[0]  # [H,W]
    else:
        img = img.permute(1, 2, 0)  # [H,W,C]
    return img.cpu().numpy()


def generate_class_truncation_grid(
    network_pkl: str,
    out_path: str = "grid_class_truncation.png",
    class_names: list[str] = ("Glioma", "Meningioma", "Pituitary"),
    truncation_psis: list[float] = [0.8, 1.0, 1.2],
    seed: int = 42,
    show: bool = True,
):
    """
    Generates a grid of images where rows represent different classes
    and columns represent different truncation_psi values.
    The grid is saved to `out_path` and optionally displayed.

    Args:
        network_pkl (str): Path to the StyleGAN2-ADA network pickle file.
        out_path (str): Path to save the generated grid image.
        class_names (list[str]): List of class names to generate images for.
                                 Each class will correspond to a row in the grid.
        truncation_psis (list[float]): List of truncation psi values.
                                       Each value will correspond to a column in the grid.
        seed (int): Random seed for reproducibility.
        show (bool): If True, displays the generated image grid using matplotlib.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] Using {device}")

    # ---------- Cargar G_ema ----------
    print(f"[Load] Loading generator from: {network_pkl}")
    with open(network_pkl, "rb") as f:
        data = load_network_pkl(f)
    G = data.get("G_ema", None) or data.get("G", None)
    if G is None:
        raise RuntimeError("No se encontró G_ema ni G en el .pkl")
    G = G.to(device).eval()

    c_dim = getattr(G, "c_dim", 0)
    z_dim = getattr(G, "z_dim", 512)
    if c_dim <= 0:
        raise RuntimeError("El generador no es condicional (c_dim=0).")

    rng = np.random.RandomState(seed)

    # Create a mapping for class names to numerical indices (0, 1, 2, etc.)
    class_to_idx = {name: i for i, name in enumerate(class_names)}

    # ---------- Preparar figura ----------
    n_rows = len(class_names)
    n_cols = len(truncation_psis)
    fig, axes = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(3 * n_cols, 3 * n_rows)
    )

    # Normalizar shape de axes para asegurar que sea 2D
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    elif n_cols == 1:
        axes = np.expand_dims(axes, axis=1)

    # Generar vectores latentes 'z' una vez para cada fila (clase)
    z_vectors = [torch.from_numpy(rng.randn(1, z_dim)).to(device) for _ in range(n_rows)]

    # ---------- Generación ----------
    with torch.no_grad():
        for row_idx, cname in enumerate(class_names):
            z = z_vectors[row_idx]  # Usar el mismo z para todas las columnas de esta fila
            class_idx = class_to_idx[cname]

            for col_idx, current_truncation_psi in enumerate(truncation_psis):
                c = torch.zeros(1, c_dim, device=device)
                c[0, class_idx] = 1.0  # La clase es fija para toda la fila

                try:
                    img = G(z, c, truncation_psi=float(current_truncation_psi), noise_mode="const")
                except TypeError:
                    img = G(z, c, truncation_psi=float(current_truncation_psi))

                img_np = to_uint8(img)

                ax = axes[row_idx, col_idx]
                if img_np.ndim == 2:
                    ax.imshow(img_np, cmap="gray")
                else:
                    ax.imshow(img_np)
                ax.axis("off")

                # Títulos para los valores de truncación en la primera fila
                if row_idx == 0:
                    ax.set_title(f"Trunc: {current_truncation_psi}", fontsize=12)

    # Add class names as titles for the rows (Y-axis labels)
    for row_idx, cname in enumerate(class_names):
        # Calculate position to center the text vertically for each row
        y_pos = 1 - ((row_idx + 0.5) / n_rows)
        fig.text(0.02, y_pos, cname, va="center", ha="center",
                 rotation="vertical", fontsize=14, transform=fig.transFigure)

    # Add a general title for the grid
    fig.suptitle(f"Generated Images (Classes vs. Truncation)", fontsize=16, y=1.02)

    plt.tight_layout(rect=[0.05, 0.0, 1.0, 1.0]) # Ajustar el layout para dejar espacio al suptitle y labels Y

    # ---------- Guardar ----------
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"[OK] Saved grid to: {out_path}")

    # ---------- Mostrar ----------
    if show:
        plt.show()

    plt.close(fig)

In [ ]:
if __name__ == "__main__":
    # Parameters for generating the grid
    network_pkl = "/content/drive/MyDrive/TESIS/StyleGan2/Modelsinge/00000-finalnuevoinge-cond-mirror-24gb-gpu-kimg1000-batch32-ada-target0.6-bg/network-snapshot-000600.pkl"
    out_path = "grid_class_truncation.png"
    class_names = ["Glioma", "Meningioma", "Pituitary"]
    truncation_psis = [ 0.8, 0.9, 1.0, 1.1,1.2, 1.3]
    seed = 9381
    show = True

    generate_class_truncation_grid(
        network_pkl=network_pkl,
        out_path=out_path,
        class_names=class_names,
        truncation_psis=truncation_psis,
        seed=seed,
        show=show,
    )

In [ ]:
if __name__ == "__main__":
    # Parameters for generating the grid
    network_pkl = "/content/drive/MyDrive/TESIS/StyleGan2/Models/00003-finalnuevo-cond-mirror-24gb-gpu-kimg1000-batch32-ada-target0.6-bg/network-snapshot-000900.pkl"
    out_path = "grid_class_truncation.png"
    class_names = ["Glioma", "Meningioma", "Pituitary"]
    truncation_psis = [ 0.8, 0.9, 1.0, 1.1,1.2, 1.3]
    seed = 886
    show = True

    generate_class_truncation_grid(
        network_pkl=network_pkl,
        out_path=out_path,
        class_names=class_names,
        truncation_psis=truncation_psis,
        seed=seed,
        show=show,
    )

**Reasoning**:
The previous code failed because the `truncation_psis` variable was commented out, leading to a `NameError`. I need to uncomment its definition so it is available when the `generate_class_truncation_grid` function is called.

